# MCP vs function calling: The same tool, two ways

**Both scripts in this folder answer "Calculate 25 + 17" with the same `add` function from `tools.py`. The only difference is where the tool lives: Inside the app (plain function calling), or behind an MCP server (MCP).**

<pre style="font-size: 0.85em; line-height: 1.35; white-space: pre; overflow-x: auto;">
 function-calling.py              mcp-calling.py
 -------------------              --------------
 schema: written by hand          schema: from tools/list
 runs add(25, 17) directly        runs session.call_tool("add", ...)
                                  and add() runs inside server.py
 one process                      two processes: client and server
</pre>

Read `../3-openai-integration/client-functions-one-question.ipynb` first. This notebook reuses its loop.

## 1. The shared tool

Both scripts use this exact function, from `tools.py`:

<div style="font-size: 0.85em">

```python
def add(a: int, b: int) -> int:
    """Add two numbers together"""
    return a + b
```

</div>

## 2. Plain function calling

You write the tool's schema yourself, and when the model asks for `add`, your own code calls `add`. There's no MCP connection, so nothing here needs to be async: The plain `OpenAI()` client does the job.

In [1]:
import json

from dotenv import load_dotenv
from openai import OpenAI

from tools import add

load_dotenv()

client = OpenAI()
model = "gpt-4o-mini"

tools = [
    {
        "type": "function",
        "function": {
            "name": "add",
            "description": "Add two numbers together",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "integer", "description": "First number"},
                    "b": {"type": "integer", "description": "Second number"},
                },
                "required": ["a", "b"],
            },
        },
    }
]

The loop is the one from `3-tools.py`. The line that matters is `result = add(args["a"], args["b"])`: A direct call to a Python function in this program.

In [2]:
messages = [{"role": "user", "content": "Calculate 25 + 17"}]

response = client.chat.completions.create(model=model, messages=messages, tools=tools)
assistant_message = response.choices[0].message
messages.append(assistant_message)

if assistant_message.tool_calls:
    for tool_call in assistant_message.tool_calls:
        name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        print("The model called:", name, args)

        result = add(args["a"], args["b"])
        messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": str(result)})

    final_response = client.chat.completions.create(model=model, messages=messages, tools=tools)
    print(final_response.choices[0].message.content)
else:
    print(assistant_message.content)

The model called: add {'a': 25, 'b': 17}
The result of \( 25 + 17 \) is \( 42 \).


## 3. The same tool behind MCP

`server.py` serves the same `add` over MCP:

<div style="font-size: 0.85em">

```python
from mcp.server.mcpserver import MCPServer

from tools import add

mcp = MCPServer("Calculator")
mcp.add_tool(add)

if __name__ == "__main__":
    mcp.run(transport="stdio")
```

</div>

`mcp.add_tool(add)` registers a function that already exists. It does the same as writing `@mcp.tool()` above it, which isn't possible here because `add` lives in another file. MCPServer builds the schema itself, from the type hints. Section 4 prints the schema the client receives.

## 4. MCP calling

`mcp-calling.py` runs the same loop, with two changes: The tools come from `tools/list`, converted to OpenAI's format as in Part 3, and the tool runs with `session.call_tool`, inside the server process. Because the MCP client is async, it uses `AsyncOpenAI` and `await`.

In [3]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from openai import AsyncOpenAI

async_client = AsyncOpenAI()
mcp_server = StdioServerParameters(command="python", args=["server.py"])

In [4]:
with open("server_log.txt", "w") as log:
    async with stdio_client(mcp_server, errlog=log) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools_result = await session.list_tools()
            tools = []
            for tool in tools_result.tools:
                tools.append(
                    {
                        "type": "function",
                        "function": {
                            "name": tool.name,
                            "description": tool.description,
                            "parameters": tool.input_schema,
                        },
                    }
                )

            messages = [{"role": "user", "content": "Calculate 25 + 17"}]

            response = await async_client.chat.completions.create(model=model, messages=messages, tools=tools)
            assistant_message = response.choices[0].message
            messages.append(assistant_message)

            if assistant_message.tool_calls:
                for tool_call in assistant_message.tool_calls:
                    name = tool_call.function.name
                    args = json.loads(tool_call.function.arguments)
                    print("The model called:", name, args)

                    result = await session.call_tool(name, args)
                    messages.append(
                        {"role": "tool", "tool_call_id": tool_call.id, "content": result.content[0].text}
                    )

                final_response = await async_client.chat.completions.create(model=model, messages=messages, tools=tools)
                print(final_response.choices[0].message.content)
            else:
                print(assistant_message.content)

The model called: add {'a': 25, 'b': 17}
The sum of 25 and 17 is 42.


The same answer, but this time `add` ran inside `server.py`, reached through two MCP requests: `tools/list` and `tools/call`. The server doesn't log each request, so its log is empty:

In [5]:
with open("server_log.txt") as log:
    print(log.read())

This is the schema the client received from `tools/list`, generated by MCPServer:

In [6]:
print(tools_result.tools[0].input_schema)

{'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object', 'title': 'addArguments'}


It has the same types and the same required arguments as the hand-written schema in section 2, but no per-argument descriptions like `"First number"`. MCPServer adds those only if the function declares them.

## 5. Side by side

- **The schema:** Written by hand in function calling. Generated by MCPServer from the type hints, and fetched with `tools/list`, in MCP.
- **Running the tool:** A direct Python call in function calling. A `tools/call` message to another process in MCP.
- **Async:** Function calling needs none. The MCP client is async, so it needs `AsyncOpenAI` and `await`.
- **Processes:** One, against two. MCP adds a child process and a JSON-RPC round trip for every call.
- **Reuse:** The hand-written tool works only inside this script. The MCP server works with any MCP host: Claude Desktop, an IDE, or another team's agent.

## 6. When to use which

**Plain function calling** fits one self-contained app with a few tools of its own, where every call should be as fast as possible, or an early prototype.

**MCP** fits when the same tools serve several apps or teams, when a ready-made server already exists, or when the tool should run somewhere else: In a container (Part 5), on another machine, or owned by another team.

## Say it out loud

> "MCP doesn't replace function calling. It moves the tool behind a standard protocol. With plain function calling, I write the schema by hand and call the function directly in my app. With MCP, the server generates the schema from the type hints, my host fetches it with `tools/list`, and the call goes to the server with `tools/call`. The model's side is identical either way. MCP costs an extra process and a round trip, so for one small app, plain function calling is simpler. MCP pays off when tools are shared, reused or run remotely."